# 08 — Goal 4: perceptual family alignment and reliability

Separates distributed main ratings, crossed four-RA reliability, and the broad two-RA labels.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main outputs: four-RA agreement, rater-stratified distributed alignment, crossed consensus alignment, and paired comparison with broad two-RA labels. Confirm scale direction and rater-folder design before running.

In [ ]:
RUN_GOAL4 = False  # Change to True only when you intend to run this stage.

if RUN_GOAL4:
    run_cli('human-qc', '--schema', 'config/human_qc_schema.yaml')
else:
    print('Goal 4 not run. Confirm the schema, four RA names, crossed Reliability folders, and label direction first.')

In [ ]:
STAGE, FIGURES, TABLES = stage_directories(Path("04_analysis") / "goal4")
source = OUTPUT / "04_analysis" / "human_qc"
agreement = read_table(source / "reliability_interrater_agreement_complete")
alignment = read_table(source / "main_distributed_rater_stratified_family_alignment")
direction = read_table(source / "two_ra_broad_direction_and_scale_audit")
save_table(agreement, TABLES, "four_ra_interrater_agreement")
save_table(alignment, TABLES, "main_rater_stratified_alignment")
save_table(direction, TABLES, "two_ra_direction_and_scale_audit")
display(direction)
display(agreement)
display(alignment)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
if not agreement.empty:
    sns.barplot(data=agreement, x="category", y="gwet_ac1_nominal", color="#59A14F", ax=axes[0])
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].set(title="Four-RA crossed reliability", xlabel="", ylabel="Gwet AC1")
matched = alignment.loc[alignment["matched_family"].fillna(False)]
if not matched.empty:
    sns.barplot(data=matched, x="human_family", y="effect", color="#4C78A8", ax=axes[1])
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].set(title="Distributed main ratings: matched-family alignment", xlabel="", ylabel="Alignment effect")
fig.tight_layout()
save_figure(fig, FIGURES, "reliability_and_family_alignment")
plt.show()

direction_ok = (
    not direction.empty
    and direction["direction"].astype(str).eq("higher_is_worse").all()
)
goal4_ready = stage_gate(
    "Goal 4",
    direction_ok and not agreement.empty,
    ([] if direction_ok else ["Broad two-RA scale direction is not confirmed as higher-is-worse."])
    + ([] if not agreement.empty else ["Four-RA reliability output is missing or blocked."]),
    "Run sensitivity summary only after reliability and scale-direction gates pass.",
)